# 03. Feature Engineering & Matrix Construction

## Overview
This notebook demonstrates feature transformations, slope derivation from elevation matrices, temperature derating calculations, missing value imputation, and final feature matrix assembly for training power output regressors.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

print("Feature engineering tools initialized!")

## 1. Feature Derivation Logic
Deriving PV temperature degradation factors and Wind Power Density proxies.

In [ ]:
np.random.seed(101)
n = 500

raw_data = pd.DataFrame({
    'solar_ghi': np.random.normal(5.5, 0.7, n),
    'wind_speed': np.random.normal(6.5, 1.2, n),
    'temp_ambient': np.random.normal(28.0, 5.0, n),
    'elevation': np.random.uniform(100, 900, n),
    'slope_deg': np.random.exponential(3.0, n)
})

# Introduce realistic missing values for testing imputation
raw_data.loc[raw_data.sample(frac=0.05).index, 'solar_ghi'] = np.nan
raw_data.loc[raw_data.sample(frac=0.05).index, 'wind_speed'] = np.nan

print(f"Raw dataset shape: {raw_data.shape}")
print(f"Missing values:\n{raw_data.isnull().sum()}")

## 2. Imputation & Feature Scaling

In [ ]:
# Impute missing values with median
imputer = SimpleImputer(strategy='median')
df_imputed = pd.DataFrame(imputer.fit_transform(raw_data), columns=raw_data.columns)

# Engineering PV Temperature Loss Penalty
# Standard Test Condition (STC) = 25°C; Loss = 0.4% per °C above STC
df_imputed['pv_temp_loss_pct'] = (df_imputed['temp_ambient'] - 25.0).clip(lower=0) * 0.4
df_imputed['pv_effective_factor'] = 1.0 - (df_imputed['pv_temp_loss_pct'] / 100.0)

# Engineering Wind Power Density (WPD) Proxy: P = 0.5 * rho * v^3 (rho ~ 1.225 kg/m3)
df_imputed['wind_power_density_wpd'] = 0.5 * 1.225 * (df_imputed['wind_speed'] ** 3)

df_imputed.head()

## 3. Scale Features for Machine Learning Inputs

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_imputed)

df_final = pd.DataFrame(X_scaled, columns=df_imputed.columns)
print("Final Feature Matrix Ready for Training!")